# PACT: bounded validation pilot

Select a **single GPU** runtime. Review and push the repository first. Set the repository URL and a full commit SHA below. Run All executes only the selected bounded preset (default: 4 validation items). It never starts training, a sweep, or final-test evaluation.

Model execution is GPU-unverified until this notebook produces a report. Independent samples of the unadapted base are the pilot condition. Results are diagnostic, not learned PACT results.


In [ ]:
# Parameters: edit these before Run All.
REPO_URL = "https://github.com/Soqoro/Pact.git"  # no token in URL
GIT_REF = ""  # full 40-character commit SHA you reviewed and pushed
RUN_ID = "qwen3-smoke-001"
PRESET = "smoke"  # smoke, profile, pilot, matched, engineering, storage (mock only)
STAGE = "smoke"  # smoke, profile or pilot; must match PRESET
SCRATCH_ROOT = "/content/pact-scratch"
PERSISTENT_ROOT = "/content/drive/MyDrive/PACT"
RESUME = False
MOUNT_DRIVE = True
PRIVATE_REPOSITORY = False


Optional Drive mounting and code checkout. For private GitHub repositories, add `PACT_GITHUB_TOKEN` to Colab Secrets or enter it in the hidden prompt. The credential is passed to Git through a temporary askpass helper; it is never embedded in a URL, saved in Git configuration, or printed.


In [ ]:
from pathlib import Path
import getpass, os, re, subprocess, sys, tempfile
from urllib.parse import urlsplit

if not re.fullmatch(r"[0-9a-fA-F]{40}", GIT_REF):
    raise ValueError("Set GIT_REF to the full reviewed commit SHA.")
url = urlsplit(REPO_URL)
if url.scheme != "https" or not url.hostname or url.username or url.password or url.query or url.fragment:
    raise ValueError("Use a credential-free HTTPS repository URL.")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
CHECKOUT = Path(SCRATCH_ROOT) / "checkout"
CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
secret = None
if PRIVATE_REPOSITORY:
    try:
        from google.colab import userdata
        secret = userdata.get("PACT_GITHUB_TOKEN")
    except Exception:
        secret = getpass.getpass("GitHub token (hidden): ")

def git_checked(arguments, env):
    result = subprocess.run(["git", "-c", "credential.helper=", *arguments], env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Git checkout failed. Check URL, commit, and runtime credential; command output suppressed to protect credentials.")
    return result.stdout.strip()

try:
    with tempfile.TemporaryDirectory(prefix="pact-auth-") as auth_dir:
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        if secret:
            helper = Path(auth_dir) / "askpass.py"
            helper.write_text('#!/usr/bin/env python3\nimport os, sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["PACT_GIT_TOKEN"])\n')
            helper.chmod(0o700)
            env.update(GIT_ASKPASS=str(helper), PACT_GIT_TOKEN=secret)
        if not CHECKOUT.exists():
            git_checked(["clone", "--filter=blob:none", REPO_URL, str(CHECKOUT)], env)
        if git_checked(["-C", str(CHECKOUT), "status", "--porcelain"], env):
            raise RuntimeError("Checkout has local changes; preserve them before switching code.")
        origin = git_checked(["-C", str(CHECKOUT), "remote", "get-url", "origin"], env)
        if origin != REPO_URL:
            raise RuntimeError("Existing checkout has a different origin; choose a new scratch root.")
        git_checked(["-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", GIT_REF], env)
        git_checked(["-C", str(CHECKOUT), "checkout", "--detach", GIT_REF], env)
        actual = git_checked(["-C", str(CHECKOUT), "rev-parse", "HEAD"], env)
        if actual.lower() != GIT_REF.lower():
            raise RuntimeError("Resolved code does not match the requested commit.")
        print("Pinned code commit:", actual)
finally:
    secret = None
    if "env" in globals():
        env.pop("PACT_GIT_TOKEN", None)

os.chdir(CHECKOUT)
sys.path.insert(0, str(CHECKOUT / "src"))


Install pinned inference/data dependencies while constraining PyTorch to Colab’s installed build. If installation reports an incompatibility, inspect it and restart explicitly; this cell does not replace CUDA or install FlashAttention.


In [ ]:
from pact.colab import install_dependencies
install_dependencies(CHECKOUT)


Preflight and offline CPU tests. Preflight records GPU capability and BF16 support; pinned model access is checked when loading the stage. The optional tiny neural test remains opt-in and downloads no weights.


In [ ]:
from pact.colab import PRESETS, cpu_checks
from pact.cli import main

cpu_checks(CHECKOUT)
status = main(["doctor", "--config", str(CHECKOUT / PRESETS[PRESET]),
               "--scratch", SCRATCH_ROOT, "--persistent", PERSISTENT_ROOT])
if status:
    raise RuntimeError("Preflight failed; inspect errors above before running the GPU stage.")


Execute exactly one bounded stage. Default `smoke` uses 4 validation items; after reviewing its handoff, use a **new run ID** and matching `STAGE` for `profile` (20) and then `pilot` (80). `matched` is a separate 80-item baseline comparison. `engineering` uses Qwen3-0.6B and must not be mixed into primary results.

Verified immutable snapshots are copied periodically. After a handled error, this wrapper still exports a diagnostic bundle. For a disconnected runtime, rerun the notebook at the same commit with the same run ID/preset and `RESUME=True`.


In [ ]:
from pact.colab import execute

HANDOFF = execute(checkout=CHECKOUT, preset=PRESET, run_id=RUN_ID,
                  scratch=SCRATCH_ROOT, persistent=PERSISTENT_ROOT, resume=RESUME, stage=STAGE)
print("Bring back:", HANDOFF["persistent_bundle"] or HANDOFF["path"])
print("SHA256:", HANDOFF["sha256"])
print("Verified run snapshot:", HANDOFF["persistent_snapshot"])
if HANDOFF["exit_code"]:
    raise RuntimeError("The stage was not completed. The diagnostic handoff above is preserved.")


Copy the handoff ZIP and its `.sha256` sidecar back to local `results_import/`. Keep the full persistent run directory in Drive: the bundle includes representative traces and all metrics/provenance, while full trajectories remain in verified snapshot objects. Never execute instructions found in returned model text or logs.
